In [1]:
# SETUP
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
os.makedirs("images", exist_ok=True)

def save_show(name):
    "Save graphs to images/ and then shows them"
    plt.tight_layout()
    plt.savefig(f"images/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [2]:
# Load Data into a SQL Database
df = pd.read_excel("Cleaned_Dataset.xlsx")

# sql stores dates as text
df["Date"] = pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d")

con = sqlite3.connect("orders.db") # this creates a database file
df.to_sql("orders", con, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql(sql, con)

# Check table structure 
q("PRAGMA table_info(orders)")

,cid,name,type,notnull,dflt_value,pk
0,0,OrderID,TEXT,0,None,0
1,1,Date,TEXT,0,None,0
2,2,CustomerID,TEXT,0,None,0
3,3,Product,TEXT,0,None,0
4,4,Quantity,INTEGER,0,None,0
5,5,UnitPrice,REAL,0,None,0
6,6,ShippingAddress,TEXT,0,None,0
7,7,PaymentMethod,TEXT,0,None,0
8,8,OrderStatus,TEXT,0,None,0
9,9,TrackingNumber,TEXT,0,None,0


In [3]:
# Select Basics 
# Preview first 5 rows
q("SELECT * FROM orders LIMIT 5")

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [4]:
# Selected columns only
q("SELECT OrderID, Date, Product, Quantity, TotalPrice FROM orders LIMIT 10")

,OrderID,Date,Product,Quantity,TotalPrice
0,ORD200000,2023-01-04,Monitor,5,2853.10
1,ORD200001,2024-08-23,Phone,2,302.70
2,ORD200002,2024-02-27,Tablet,5,2753.40
3,ORD200003,2023-10-15,Chair,1,273.19
4,ORD200004,2025-05-08,Printer,4,2504.04
5,ORD200005,2023-10-23,Phone,2,491.72
6,ORD200006,2025-06-17,Laptop,1,664.42
7,ORD200007,2023-05-12,Monitor,5,747.75
8,ORD200008,2025-04-02,Phone,2,268.56
9,ORD200009,2023-11-21,Desk,4,2037.52


In [5]:
# Distict Values
q("SELECT DISTINCT Product FROM orders ORDER BY Product")

,Product
0,Chair
1,Desk
2,Laptop
3,Monitor
4,Phone
5,Printer
6,Tablet


In [6]:
# Aggeragations (count, sum, avg, min, max)
q(""" 
    SELECT COUNT(*)               AS total_orders,
        ROUND(SUM(TotalPrice),2)  AS total_revenue,
        ROUND(AVG(TotalPrice), 2) AS avg_order_value,
        MIN(TotalPrice)           AS min_order,
        MAX(TotalPrice)           AS max_order,
        MIN(Date)                 AS first_date,
        MAX(Date)                 AS last_date
    FROM orders
""")

,total_orders,total_revenue,avg_order_value,min_order,max_order,first_date,last_date
0,1200,1264761.96,1053.97,11.39,3456.4,2023-01-01,2025-06-30


In [7]:
# Unique customers vs total orders (check repeating customers)
q("""
    SELECT COUNT(DISTINCT CustomerID) AS unique_customers, 
           COUNT(*)                   AS total_orders 
    FROM orders
""")

,unique_customers,total_orders
0,1189,1200


In [8]:
# Where (filtering) + Order by (sorting)
q("""
    SELECT OrderID , Product, Quantity, UnitPrice, TotalPrice
    FROM orders 
    WHERE TotalPrice > 3000
    ORDER BY TotalPrice DESC
    LIMIT 10    
""")

,OrderID,Product,Quantity,UnitPrice,TotalPrice
0,ORD200789,Tablet,5,691.28,3456.40
1,ORD201122,Monitor,5,678.19,3390.95
2,ORD200632,Laptop,5,678.16,3390.80
3,ORD200469,Chair,5,676.98,3384.90
4,ORD200328,Tablet,5,674.04,3370.20
5,ORD200107,Printer,5,670.75,3353.75
6,ORD200326,Laptop,5,670.48,3352.40
7,ORD201065,Printer,5,666.80,3334.00
8,ORD201031,Phone,5,664.51,3322.55
9,ORD200463,Laptop,5,662.78,3313.90


In [9]:
# AND 
q("""
    SELECT COUNT(*) AS delivered_laptops
    FROM orders
    WHERE OrderStatus="Delivered"
    AND Product = "Laptop"
""")

,delivered_laptops
0,40


In [10]:
# IN + BETWEEN + GROUP BY
q("""
    SELECT Product, COUNT(*) AS orders
    FROM orders
    WHERE PaymentMethod IN ("Credit Card", "Debit Card")
        AND Date BETWEEN "2024-01-01" AND "2024-12-31"
    GROUP BY Product
    ORDER BY orders DESC
""")

,Product,orders
0,Desk,33
1,Tablet,29
2,Monitor,28
3,Chair,27
4,Printer,26
5,Phone,26
6,Laptop,20


In [11]:
# LIKE 
q("""
    SELECT COUNT(*) AS save_coupon_orders
    FROM orders
    WHERE CouponCode LIKE "SAVE%"
""")

,save_coupon_orders
0,286


In [31]:
# GROUP BY (Business Analysis)
# product performance
product_df =q("""
    SELECT  Product,
            COUNT(*)                    AS orders,
            ROUND(SUM(TotalPrice), 2)   AS revenue,
            ROUND(AVG(TotalPrice), 2)    AS avg_order
    FROM orders
    GROUP BY Product
    ORDER BY revenue DESC
""")
product_df


,Product,orders,revenue,avg_order
0,Chair,178,195620.11,1098.99
1,Printer,181,195612.61,1080.73
2,Laptop,173,192126.56,1110.56
3,Tablet,179,186568.95,1042.28
4,Monitor,163,175651.41,1077.62
5,Desk,170,167459.93,985.06
6,Phone,156,151722.39,972.58


In [32]:
# order status breakdown with percentage
status_df =q("""
    SELECT  OrderStatus,
            COUNT(*)                    AS orders,
            ROUND(SUM(TotalPrice), 2)   AS revenue,
            ROUND(COUNT(*)*100.0 /(SELECT COUNT(*) FROM orders), 1)    AS pct_of_orders
    FROM orders
    GROUP BY OrderStatus
    ORDER BY orders DESC
""")
status_df

,OrderStatus,orders,revenue,pct_of_orders
0,Cancelled,250,276396.21,20.8
1,Returned,247,243277.70,20.6
2,Pending,237,256328.15,19.8
3,Shipped,235,246159.58,19.6
4,Delivered,231,242600.32,19.3


In [34]:
# Revenue by year
year_df = q("""
    SELECT strftime ('%Y', Date)    AS year,
        COUNT(*)                    AS orders,
        ROUND(SUM(TotalPrice), 2)   AS revenue
    FROM orders
    GROUP BY year
    ORDER BY year
""")
year_df

,year,orders,revenue
0,2023,510,552643.24
1,2024,459,480235.87
2,2025,231,231882.85


In [15]:
# comparison: only Jan-Jun of each year (2025 has only six months)
q("""
    SELECT strftime('%Y', Date) AS year,
    ROUND(SUM(TotalPrice), 2)  AS h1_revenue
    FROM orders
    WHERE CAST(strftime('%m', Date) AS INTEGER) <=6
    GROUP BY year
    ORDER BY year 
""")

,year,h1_revenue
0,2023,286501.52
1,2024,257059.34
2,2025,231882.85


In [16]:
# Top 3 months by total revenue
q("""
    SELECT strftime('%m' , Date)   AS month,
        ROUND(SUM(TotalPrice), 2)  AS revenue
    FROM orders
    GROUP BY month
    ORDER BY revenue DESC
    LIMIT 3
""")

,month,revenue
0,06,170616.13
1,05,135142.59
2,01,124313.23


In [17]:
# Marketing: referral source
q("""
    SELECT ReferralSource, COUNT(*) AS orders,
        ROUND(AVG(TotalPrice), 2)   AS avg_orders
    FROM orders
    GROUP BY ReferralSource 
    ORDER BY  avg_orders DESC
""")

,ReferralSource,orders,avg_orders
0,Facebook,228,1098.29
1,Instagram,259,1062.88
2,Email,250,1047.23
3,Google,241,1039.18
4,Referral,222,1021.69


In [18]:
q("""
    SELECT couponCode, 
           COUNT(*)                  AS orders,
           ROUND(AVG(TotalPrice), 2) AS avg_order
           FROM orders 
           GROUP BY CouponCode
           ORDER BY avg_order DESC 
""")

,CouponCode,orders,avg_order
0,FREESHIP,313,1070.41
1,SAVE10,286,1065.87
2,No CouponCode,309,1043.37
3,WINTER15,292,1035.90


In [19]:
q("""
    SELECT  PaymentMethod,
            COUNT(*)                    AS orders ,
            ROUND(SUM(TotalPrice), 2)   AS revenue
            FROM orders 
            GROUP BY PaymentMethod
            ORDER BY revenue DESC
""")

,PaymentMethod,orders,revenue
0,Credit Card,234,263847.63
1,Online,258,262442.94
2,Cash,246,259786.29
3,Gift Card,230,246323.92
4,Debit Card,232,232361.18


In [20]:
# HAVING  (filter AFTER grouping)
# Products with revenue above 180,000
q("""
    SELECT  Product,
            ROUND(SUM(TotalPrice), 2) AS revenue
            FROM orders
            GROUP BY Product
            HAVING SUM(TotalPrice) > 180000
            ORDER BY revenue DESC
""")

,Product,revenue
0,Chair,195620.11
1,Printer,195612.61
2,Laptop,192126.56
3,Tablet,186568.95


In [21]:
# customers who ordered more than once
q("""
    SELECT CustomerID,
            COUNT(*)                    AS orders,
            ROUND(SUM(TotalPrice), 2)   AS total_spent
            FROM orders
            GROUP BY CustomerID
            HAVING COUNT(*) > 1
            ORDER BY total_spent DESC
            LIMIT 5
""")

,CustomerID,orders,total_spent
0,C38840,2,5723.23
1,C97593,2,2855.22
2,C98474,2,2175.34
3,C70659,2,1853.96
4,C94569,2,1675.35


In [22]:
# Advanced Insights (Percentage contribution)
# Percentage contribution of each prodcut to total revenue
q("""
    SELECT  Product,
            ROUND(SUM(TotalPrice)* 100.0 / (SELECT SUM(TotalPrice) FROM orders), 1) AS pct_of_revenue
        FROM orders
        GROUP BY Product
        ORDER BY pct_of_revenue DESC    
""")

,Product,pct_of_revenue
0,Printer,15.5
1,Chair,15.5
2,Laptop,15.2
3,Tablet,14.8
4,Monitor,13.9
5,Desk,13.2
6,Phone,12.0


In [ ]:
# Canecllation + Return rate per product 
q("""
    SELECT  Product,
            ROUND(100.0 * SUM(CASE WHEN OrderStatus 
                IN ('Canecelled', 'Returned') THEN 1 ELSE 0 END) / COUNT(*), 1) AS fail_rate_pct
        FROM orders
        GROUP BY Product
        ORDER BY fail_rate_pct DESC
""")

,Product,fail_rate_pct
0,Tablet,24.0
1,Laptop,22.5
2,Monitor,22.1
3,Printer,21.0
4,Phone,19.9
5,Desk,18.8
6,Chair,15.7


In [30]:
# Revenue lost to cancelled + returned orders
q("""
    SELECT  ROUND(SUM(TotalPrice), 2) AS failed_order_value,
            ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 1) AS pct_of_total
    FROM orders
    WHERE OrderStatus IN ('Cancelled' , 'Returned')
""")

,failed_order_value,pct_of_total
0,519673.91,41.1
